# WiSig — Module 2 v2 (Soft-routing, protocol-suite aligned)

This notebook upgrades the protocol-suite aligned Module 2 into a **soft-routing v2** version intended to serve as a more reliable upstream router for Module 3.

Main changes relative to the previous aligned version:
- keep the same **protocol-suite evaluation**: RX {3-9, 6-6, 9-3} × TX {2-4, 3-3, 4-2}
- replace the previous **hard sequential gate** with a **decoupled soft router**
- compute separate **knownness** and **driftness** probabilities, then form:
  - `p_stable = p_known * (1 - p_drift_cond)`
  - `p_drift = p_known * p_drift_cond`
  - `p_unknown = 1 - p_known`
- enrich the identity score with an additional **prototype-margin** term
- export both traditional hard metrics and the underlying **soft routing statistics** for downstream Module 3 analysis

Interpretation:
- this v2 notebook does **not** change the feature backbone first
- it focuses on the bottleneck exposed by Module 2 evaluation: **routing/calibration**, especially drift recall vs unknown rejection trade-off



In [1]:
import os, json, time
import itertools
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path, dataset_name):
    with open(dataset_path + dataset_name + '.pkl', 'rb') as f:
        dataset = pickle.load(f)
    return dataset

compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# ---- subset sizes ----
TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4

# ---- protocol suite aligned with Module 3 ----
TX_SPLIT_REPEATS = 5

RX_PROTOCOL_LIBRARY = {
    "3-9": dict(source_rx_num=3, drift_rx_num=9),
    "6-6": dict(source_rx_num=6, drift_rx_num=6),
    "9-3": dict(source_rx_num=9, drift_rx_num=3),
}
TX_PROTOCOL_LIBRARY = {
    "2-4": dict(known_tx_num=2, unknown_tx_num=4),
    "3-3": dict(known_tx_num=3, unknown_tx_num=3),
    "4-2": dict(known_tx_num=4, unknown_tx_num=2),
}

SELECTED_RX_PROTOCOLS = ["3-9", "6-6", "9-3"]
SELECTED_TX_PROTOCOLS = ["2-4", "3-3", "4-2"]
SELECTED_PROTOCOL_COMBOS = None

# ---- protocol ----
TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

# ---- signals ----
Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

# ---- training ----
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3

# ---- dims ----
D_DIM = 32

# ---- Sid fusion ----
FUSION_LAM = 0.5
SID_MARGIN_LAM = 0.75

# ---- soft router calibration ----
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05
SOFT_ROUTE_UNKNOWN_PRIOR = 1.0
SOFT_ROUTE_DRIFT_PRIOR = 1.0
MIN_TEMP = 0.15

# ---- outputs ----
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"../owdc_results//results_wisig_module2_v2_soft_router/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED,
    TX_TOTAL_USE=TX_TOTAL_USE,
    RX_TOTAL_USE=RX_TOTAL_USE,
    DAY_TOTAL_USE=DAY_TOTAL_USE,
    TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    RX_PROTOCOL_LIBRARY=RX_PROTOCOL_LIBRARY,
    TX_PROTOCOL_LIBRARY=TX_PROTOCOL_LIBRARY,
    SELECTED_RX_PROTOCOLS=SELECTED_RX_PROTOCOLS,
    SELECTED_TX_PROTOCOLS=SELECTED_TX_PROTOCOLS,
    SELECTED_PROTOCOL_COMBOS=SELECTED_PROTOCOL_COMBOS,
    TRAIN_DATES=TRAIN_DATES,
    TEST_DATES_RX=TEST_DATES_RX,
    TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ,
    D_FROM=D_FROM,
    MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES,
    DROPOUT=DROPOUT,
    D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM,
    SID_MARGIN_LAM=SID_MARGIN_LAM,
    FRR_TARGET=FRR_TARGET,
    FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
    SOFT_ROUTE_UNKNOWN_PRIOR=SOFT_ROUTE_UNKNOWN_PRIOR,
    SOFT_ROUTE_DRIFT_PRIOR=SOFT_ROUTE_DRIFT_PRIOR,
    MIN_TEMP=MIN_TEMP,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("RUN_DIR:", RUN_DIR)



Device: cuda
RUN_DIR: ../owdc_results//results_wisig_module2_v2_soft_router/run_20260318_155434


In [2]:
# =========================
# Data utilities
# =========================
def get_idx(lst, val): 
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

In [3]:
# =========================
# Protocol suite construction
# =========================
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)

def build_unique_tx_splits(tx_use, known_tx_num, n_splits, seed=42):
    all_known = [tuple(c) for c in itertools.combinations(list(tx_use), known_tx_num)]
    if n_splits > len(all_known):
        raise ValueError(
            f"Requested TX_SPLIT_REPEATS={n_splits}, but only {len(all_known)} unique TX splits "
            f"exist for C({len(tx_use)},{known_tx_num})."
        )
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(all_known))
    tx_splits = []
    for local_id, idx in enumerate(order[:n_splits], start=1):
        known = list(all_known[idx])
        unknown = [tx for tx in tx_use if tx not in known]
        tx_splits.append({
            "split_id": local_id,
            "known": known,
            "unknown": unknown,
        })
    return tx_splits

def build_protocol_specs(
    tx_use, rx_use,
    selected_rx_protocols, selected_tx_protocols,
    rx_protocol_library, tx_protocol_library,
    tx_split_repeats, seed=42, explicit_protocol_combos=None
):
    if explicit_protocol_combos is None:
        combo_pairs = [(rx_tag, tx_tag) for rx_tag in selected_rx_protocols for tx_tag in selected_tx_protocols]
    else:
        combo_pairs = list(explicit_protocol_combos)

    protocol_specs = []
    for combo_idx, (rx_tag, tx_tag) in enumerate(combo_pairs, start=1):
        if rx_tag not in rx_protocol_library:
            raise KeyError(f"Unknown RX protocol tag: {rx_tag}")
        if tx_tag not in tx_protocol_library:
            raise KeyError(f"Unknown TX protocol tag: {tx_tag}")

        rx_cfg = dict(rx_protocol_library[rx_tag])
        tx_cfg = dict(tx_protocol_library[tx_tag])

        src_n = int(rx_cfg["source_rx_num"])
        drift_n = int(rx_cfg["drift_rx_num"])
        known_n = int(tx_cfg["known_tx_num"])
        unknown_n = int(tx_cfg["unknown_tx_num"])

        if src_n + drift_n != len(rx_use):
            raise ValueError(
                f"RX protocol {rx_tag} expects source+drift={src_n + drift_n}, "
                f"but len(RX_USE)={len(rx_use)}."
            )
        if known_n + unknown_n != len(tx_use):
            raise ValueError(
                f"TX protocol {tx_tag} expects known+unknown={known_n + unknown_n}, "
                f"but len(TX_USE)={len(tx_use)}."
            )

        rng_rx = np.random.RandomState(seed + 1000 * combo_idx + 17 * src_n + drift_n)
        source_rxs = list(rng_rx.choice(list(rx_use), size=src_n, replace=False))
        source_rxs.sort()
        drift_rx = [r for r in rx_use if r not in source_rxs]

        tx_splits = build_unique_tx_splits(
            tx_use=tx_use,
            known_tx_num=known_n,
            n_splits=tx_split_repeats,
            seed=seed + 5000 + 97 * combo_idx,
        )

        protocol_specs.append(dict(
            protocol_tag=f"RX{rx_tag}_TX{tx_tag}",
            rx_protocol=rx_tag,
            tx_protocol=tx_tag,
            source_rx_num=src_n,
            drift_rx_num=drift_n,
            known_tx_num=known_n,
            unknown_tx_num=unknown_n,
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            tx_splits=tx_splits,
        ))
    return protocol_specs

PROTOCOL_SPECS = build_protocol_specs(
    TX_USE, RX_USE,
    SELECTED_RX_PROTOCOLS, SELECTED_TX_PROTOCOLS,
    RX_PROTOCOL_LIBRARY, TX_PROTOCOL_LIBRARY,
    tx_split_repeats=TX_SPLIT_REPEATS,
    seed=SEED + 777,
    explicit_protocol_combos=SELECTED_PROTOCOL_COMBOS,
)

with open(os.path.join(RUN_DIR, "protocol_specs.json"), "w", encoding="utf-8") as f:
    json.dump(PROTOCOL_SPECS, f, indent=2)

print("Selected protocol combinations:", len(PROTOCOL_SPECS))
for spec in PROTOCOL_SPECS:
    print(
        f"[{spec['protocol_tag']}] "
        f"SOURCE_RXS={spec['source_rxs']} | DRIFT_RX={spec['drift_rx']} | "
        f"TXsplits={len(spec['tx_splits'])}"
    )


TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
Selected protocol combinations: 9
[RX3-9_TX2-4] SOURCE_RXS=[np.str_('19-2'), np.str_('2-1'), np.str_('20-1')] | DRIFT_RX=['1-1', '1-19', '14-7', '18-2', '2-19', '3-19', '7-14', '7-7', '8-8'] | TXsplits=5
[RX3-9_TX3-3] SOURCE_RXS=[np.str_('2-1'), np.str_('2-19'), np.str_('20-1')] | DRIFT_RX=['1-1', '1-19', '14-7', '18-2', '19-2', '3-19', '7-14', '7-7', '8-8'] | TXsplits=5
[RX3-9_TX4-2] SOURCE_RXS=[np.str_('19-2'), np.str_('2-1'), np.str_('20-1')] | DRIFT_RX=['1-1', '1-19', '14-7', '18-2', '2-19', '3-19', '7-14', '7-7', '8-8'] | TXsplits=5
[RX6-6_TX2-4] SOURCE_RXS=[np.str_('1-1'), np.str_('14-7'), np.str_('2-1'), np.str_('3-19'), np.str_('7-14'), np.str_('7-7')] | DRIFT_RX=['1-19', '18-2', '19-2', '2-19', '20-1', '8-8'] | TXsplits=5
[RX6-6_TX3-3] SOURCE_RXS=[np.st

In [4]:
# =========================
# Model: ResNet18-1D
# =========================
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [5]:
# =========================
# Train / inference helpers
# =========================
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss / max(1, total), total_correct / max(1, total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all, 0), np.concatenate(Z_all, 0)

def roc_auc(y_true, score):
    y_true = np.asarray(y_true)
    score = np.asarray(score)
    if len(np.unique(y_true)) < 2:
        return float('nan')
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

def l2norm(x, axis=1, eps=1e-12):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        P[k] = ZN[y == k].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    return l2norm(Z, axis=1) @ P.T

def sid_EmbMaha(Z, mu_z, var_z):
    dif = Z[:, None, :] - mu_z[None, :, :]
    dist = np.sum((dif * dif) / (var_z[None, :, :] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train == k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_Energy(logits):
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits - m).sum(axis=1, keepdims=True) + 1e-12)).squeeze(1).astype(np.float32)

def sid_ProtoMargin(cosine_scores):
    cs = np.asarray(cosine_scores, dtype=np.float32)
    if cs.shape[1] == 1:
        return cs[:, 0].astype(np.float32)
    top2 = np.partition(cs, kth=cs.shape[1] - 2, axis=1)[:, -2:]
    return (top2[:, 1] - top2[:, 0]).astype(np.float32)

def zscore(x, ref):
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if std < 1e-12:
        std = 1.0
    return ((x - mu) / std).astype(np.float32)

def robust_scale(x, min_scale=MIN_TEMP):
    x = np.asarray(x, dtype=np.float32)
    q75, q25 = np.quantile(x, [0.75, 0.25])
    iqr = float(q75 - q25)
    std = float(np.std(x))
    scale = max(iqr / 1.349 if iqr > 1e-8 else 0.0, std, min_scale)
    return float(scale)

def sigmoid_np(x):
    x = np.clip(x, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-x))



In [6]:
# =========================
# Domain (Sdom): V1_mixNLL
# =========================
def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps=[]
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum() + 1e-12) - np.log(R)
        out[i] = float(-loglik)  # higher => more drift
    return out

In [7]:
# =========================
# Build datasets
# =========================
def build_source_train(compact_dataset, KNOWN_TX, source_rxs):
    X_list, y_list, D_list = [], [], []
    rx_id_list = []
    for tx in KNOWN_TX:
        for rx in source_rxs:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]
                Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz)
                y_list.append(y)
                D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    if not X_list:
        raise ValueError("Empty source training set. Check protocol configuration.")
    X = np.concatenate(X_list, 0).astype(np.float32)
    y = np.concatenate(y_list, 0).astype(np.int64)
    D = np.concatenate(D_list, 0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list, 0).astype(np.int64)
    return X, y, D, rx_id

def build_eval_set(compact_dataset, txs, rxs, days):
    X_list, D_list = [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]
                Xd = Xd[:n]
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz)
                D_list.append(Df)
    if not X_list:
        raise ValueError("Empty evaluation set. Check protocol configuration.")
    X = np.concatenate(X_list, 0).astype(np.float32)
    D = np.concatenate(D_list, 0).astype(np.float32)
    return X, D


In [8]:
# =========================
# Soft router + plots
# =========================
def calibrate_soft_router(Sid_stable, Sdom_stable, frr_target=0.05, fdr_target=0.05):
    tau_id = float(np.quantile(Sid_stable, frr_target))
    tau_dom = float(np.quantile(Sdom_stable, 1.0 - fdr_target))
    temp_id = robust_scale(Sid_stable, min_scale=MIN_TEMP)
    temp_dom = robust_scale(Sdom_stable, min_scale=MIN_TEMP)
    return tau_id, tau_dom, temp_id, temp_dom

def soft_route_probs(
    Sid,
    Sdom,
    tau_id,
    tau_dom,
    temp_id,
    temp_dom,
    unknown_prior=1.0,
    drift_prior=1.0,
):
    p_known = sigmoid_np((Sid - tau_id) / max(temp_id, 1e-6))
    p_drift_cond = sigmoid_np((Sdom - tau_dom) / max(temp_dom, 1e-6))

    p_unknown = np.clip((1.0 - p_known) * unknown_prior, 1e-8, None)
    p_stable = np.clip(p_known * (1.0 - p_drift_cond), 1e-8, None)
    p_drift = np.clip(p_known * p_drift_cond * drift_prior, 1e-8, None)

    probs = np.stack([p_stable, p_drift, p_unknown], axis=1).astype(np.float32)
    probs /= np.clip(probs.sum(axis=1, keepdims=True), 1e-8, None)
    pred = np.argmax(probs, axis=1).astype(np.int64)
    aux = {
        "p_known": p_known.astype(np.float32),
        "p_drift_cond": p_drift_cond.astype(np.float32),
    }
    return probs, pred, aux

def plot_gate_scatter(Sid, Sdom, gt, tau_id, tau_dom, out_png, max_points=20000):
    n = len(Sid)
    if n > max_points:
        idx = np.random.RandomState(SEED).choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6, 5))
    plt.scatter(Sid[idx], Sdom[idx], s=3, c=gt[idx], alpha=0.5)
    plt.axvline(tau_id, linestyle="--")
    plt.axhline(tau_dom, linestyle="--")
    plt.xlabel("S_id_soft (higher=more known)")
    plt.ylabel("S_dom (higher=more drift)")
    plt.title("Module2 v2 soft router (0=stable,1=drift,2=unknown)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def plot_confmat(cm, out_png, title):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xticks([0, 1, 2], ["Stable", "Drift", "Unknown"])
    plt.yticks([0, 1, 2], ["Stable", "Drift", "Unknown"])
    plt.xlabel("Pred")
    plt.ylabel("GT")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()



In [9]:
# =========================
# Main loop
# =========================
def run_one_split(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id, KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx):
    protocol_dir = os.path.join(RUN_DIR, protocol_tag)
    split_dir = os.path.join(protocol_dir, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)

    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "protocol_tag": protocol_tag,
                "rx_protocol": rx_protocol_tag,
                "tx_protocol": tx_protocol_tag,
                "KNOWN_TX": KNOWN_TX,
                "UNKNOWN_TX": UNKNOWN_TX,
                "SOURCE_RXS": source_rxs,
                "DRIFT_RX": drift_rx,
            },
            f,
            indent=2,
        )

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX, source_rxs)
    K = len(KNOWN_TX)

    X_drRX, D_drRX = build_eval_set(compact_dataset, KNOWN_TX, drift_rx, TEST_DATES_RX)
    X_drDAY, D_drDAY = build_eval_set(compact_dataset, KNOWN_TX, source_rxs, TEST_DATES_DAY)

    X_u_in, D_u_in = build_eval_set(compact_dataset, UNKNOWN_TX, source_rxs, TEST_DATES_RX)
    X_u_drRX, D_u_drRX = build_eval_set(compact_dataset, UNKNOWN_TX, drift_rx, TEST_DATES_RX)
    X_u_drDAY, D_u_drDAY = build_eval_set(compact_dataset, UNKNOWN_TX, source_rxs, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000 * split_id + len(source_rxs))
    rows = []

    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rng = np.random.RandomState(SEED + 1000 * split_id + 100 * len(source_rxs) + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(ArrayDataset(X_src[idx_val], y_src[idx_val]), batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val = -1.0
        best_state = None
        patience = 0
        hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        for ep in range(1, EPOCHS + 1):
            tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=opt)
            va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
            hist["train_loss"].append(tr_loss)
            hist["train_acc"].append(tr_acc)
            hist["val_loss"].append(va_loss)
            hist["val_acc"].append(va_acc)

            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
        with open(os.path.join(fold_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(hist, f, indent=2)
        model.load_state_dict(best_state)

        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        acc = float((np.argmax(logits_A, 1) == y_src[idx_te]).mean())
        D_A = D_src[idx_te]

        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        mu_z, var_z = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)
        P = prototypes(Z_tr, y_src[idx_train], K)

        logits_B, Z_B = infer_logits_embed(model, X_drRX, batch=512)
        logits_C, Z_C = infer_logits_embed(model, X_drDAY, batch=512)
        logits_D, Z_D = infer_logits_embed(model, X_u_in, batch=512)
        logits_E, Z_E = infer_logits_embed(model, X_u_drRX, batch=512)
        logits_F, Z_F = infer_logits_embed(model, X_u_drDAY, batch=512)

        SidA_emb = sid_EmbMaha(Z_A, mu_z, var_z)
        SidB_emb = sid_EmbMaha(Z_B, mu_z, var_z)
        SidC_emb = sid_EmbMaha(Z_C, mu_z, var_z)
        SidD_emb = sid_EmbMaha(Z_D, mu_z, var_z)
        SidE_emb = sid_EmbMaha(Z_E, mu_z, var_z)
        SidF_emb = sid_EmbMaha(Z_F, mu_z, var_z)

        SidA_en = sid_Energy(logits_A)
        SidB_en = sid_Energy(logits_B)
        SidC_en = sid_Energy(logits_C)
        SidD_en = sid_Energy(logits_D)
        SidE_en = sid_Energy(logits_E)
        SidF_en = sid_Energy(logits_F)

        CosA = cosine_to_proto(Z_A, P)
        CosB = cosine_to_proto(Z_B, P)
        CosC = cosine_to_proto(Z_C, P)
        CosD = cosine_to_proto(Z_D, P)
        CosE = cosine_to_proto(Z_E, P)
        CosF = cosine_to_proto(Z_F, P)

        SidA_margin = sid_ProtoMargin(CosA)
        SidB_margin = sid_ProtoMargin(CosB)
        SidC_margin = sid_ProtoMargin(CosC)
        SidD_margin = sid_ProtoMargin(CosD)
        SidE_margin = sid_ProtoMargin(CosE)
        SidF_margin = sid_ProtoMargin(CosF)

        zA_emb = zscore(SidA_emb, SidA_emb)
        zB_emb = zscore(SidB_emb, SidA_emb)
        zC_emb = zscore(SidC_emb, SidA_emb)
        zD_emb = zscore(SidD_emb, SidA_emb)
        zE_emb = zscore(SidE_emb, SidA_emb)
        zF_emb = zscore(SidF_emb, SidA_emb)

        zA_en = zscore(SidA_en, SidA_en)
        zB_en = zscore(SidB_en, SidA_en)
        zC_en = zscore(SidC_en, SidA_en)
        zD_en = zscore(SidD_en, SidA_en)
        zE_en = zscore(SidE_en, SidA_en)
        zF_en = zscore(SidF_en, SidA_en)

        zA_margin = zscore(SidA_margin, SidA_margin)
        zB_margin = zscore(SidB_margin, SidA_margin)
        zC_margin = zscore(SidC_margin, SidA_margin)
        zD_margin = zscore(SidD_margin, SidA_margin)
        zE_margin = zscore(SidE_margin, SidA_margin)
        zF_margin = zscore(SidF_margin, SidA_margin)

        Sid_A = (zA_emb + FUSION_LAM * zA_en + SID_MARGIN_LAM * zA_margin).astype(np.float32)
        Sid_B = (zB_emb + FUSION_LAM * zB_en + SID_MARGIN_LAM * zB_margin).astype(np.float32)
        Sid_C = (zC_emb + FUSION_LAM * zC_en + SID_MARGIN_LAM * zC_margin).astype(np.float32)
        Sid_D = (zD_emb + FUSION_LAM * zD_en + SID_MARGIN_LAM * zD_margin).astype(np.float32)
        Sid_E = (zE_emb + FUSION_LAM * zE_en + SID_MARGIN_LAM * zE_margin).astype(np.float32)
        Sid_F = (zF_emb + FUSION_LAM * zF_en + SID_MARGIN_LAM * zF_margin).astype(np.float32)

        kA = np.argmax(CosA, 1).astype(np.int64)
        kB = np.argmax(CosB, 1).astype(np.int64)
        kC = np.argmax(CosC, 1).astype(np.int64)
        kD = np.argmax(CosD, 1).astype(np.int64)
        kE = np.argmax(CosE, 1).astype(np.int64)
        kF = np.argmax(CosF, 1).astype(np.int64)

        source_rx_ids = sorted({RX_USE.index(r) for r in source_rxs})
        models_kr, fb = fit_device_rx_models(
            D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20
        )

        Sdom_A = sdom_V1_mixNLL(D_A, kA, models_kr, fb, source_rx_ids)
        Sdom_B = sdom_V1_mixNLL(D_drRX, kB, models_kr, fb, source_rx_ids)
        Sdom_C = sdom_V1_mixNLL(D_drDAY, kC, models_kr, fb, source_rx_ids)
        Sdom_D = sdom_V1_mixNLL(D_u_in, kD, models_kr, fb, source_rx_ids)
        Sdom_E = sdom_V1_mixNLL(D_u_drRX, kE, models_kr, fb, source_rx_ids)
        Sdom_F = sdom_V1_mixNLL(D_u_drDAY, kF, models_kr, fb, source_rx_ids)

        tau_id, tau_dom, temp_id, temp_dom = calibrate_soft_router(Sid_A, Sdom_A, FRR_TARGET, FALSE_DRIFT_TARGET)
        with open(os.path.join(fold_dir, "thresholds.json"), "w", encoding="utf-8") as f:
            json.dump(
                {
                    "tau_id": tau_id,
                    "tau_dom": tau_dom,
                    "temp_id": temp_id,
                    "temp_dom": temp_dom,
                    "unknown_prior": SOFT_ROUTE_UNKNOWN_PRIOR,
                    "drift_prior": SOFT_ROUTE_DRIFT_PRIOR,
                },
                f,
                indent=2,
            )

        probs_A, pred_A, aux_A = soft_route_probs(Sid_A, Sdom_A, tau_id, tau_dom, temp_id, temp_dom, SOFT_ROUTE_UNKNOWN_PRIOR, SOFT_ROUTE_DRIFT_PRIOR)
        probs_B, pred_B, aux_B = soft_route_probs(Sid_B, Sdom_B, tau_id, tau_dom, temp_id, temp_dom, SOFT_ROUTE_UNKNOWN_PRIOR, SOFT_ROUTE_DRIFT_PRIOR)
        probs_C, pred_C, aux_C = soft_route_probs(Sid_C, Sdom_C, tau_id, tau_dom, temp_id, temp_dom, SOFT_ROUTE_UNKNOWN_PRIOR, SOFT_ROUTE_DRIFT_PRIOR)
        probs_D, pred_D, aux_D = soft_route_probs(Sid_D, Sdom_D, tau_id, tau_dom, temp_id, temp_dom, SOFT_ROUTE_UNKNOWN_PRIOR, SOFT_ROUTE_DRIFT_PRIOR)
        probs_E, pred_E, aux_E = soft_route_probs(Sid_E, Sdom_E, tau_id, tau_dom, temp_id, temp_dom, SOFT_ROUTE_UNKNOWN_PRIOR, SOFT_ROUTE_DRIFT_PRIOR)
        probs_F, pred_F, aux_F = soft_route_probs(Sid_F, Sdom_F, tau_id, tau_dom, temp_id, temp_dom, SOFT_ROUTE_UNKNOWN_PRIOR, SOFT_ROUTE_DRIFT_PRIOR)

        Sid_all = np.concatenate([Sid_A, Sid_B, Sid_C, Sid_D, Sid_E, Sid_F])
        Sdom_all = np.concatenate([Sdom_A, Sdom_B, Sdom_C, Sdom_D, Sdom_E, Sdom_F])
        gt = np.concatenate([
            np.zeros_like(Sid_A, dtype=np.int64),
            np.ones_like(Sid_B, dtype=np.int64),
            np.ones_like(Sid_C, dtype=np.int64),
            np.full_like(Sid_D, 2, dtype=np.int64),
            np.full_like(Sid_E, 2, dtype=np.int64),
            np.full_like(Sid_F, 2, dtype=np.int64),
        ])
        pred_all = np.concatenate([pred_A, pred_B, pred_C, pred_D, pred_E, pred_F])
        cm = confusion_matrix(gt, pred_all, labels=[0, 1, 2])

        plot_gate_scatter(Sid_all, Sdom_all, gt, tau_id, tau_dom, os.path.join(fold_dir, "gate_scatter.png"))
        plot_confmat(cm, os.path.join(fold_dir, "gate_confmat.png"), "3-class soft-routing confusion")

        unknown_preds = np.concatenate([pred_D, pred_E, pred_F])
        drift_preds = np.concatenate([pred_B, pred_C])

        stable_acc_gate = float((pred_A == 0).mean())
        drift_acc_rx = float((pred_B == 1).mean())
        drift_acc_day = float((pred_C == 1).mean())
        drift_acc_all = float((drift_preds == 1).mean())

        unknown_reject_in = float((pred_D == 2).mean())
        unknown_reject_rx = float((pred_E == 2).mean())
        unknown_reject_day = float((pred_F == 2).mean())
        unknown_reject_all = float((unknown_preds == 2).mean())

        FRR_stable = float((pred_A != 0).mean())
        false_drift_stable = float((pred_A == 1).mean())
        false_unknown_stable = float((pred_A == 2).mean())

        FAR_unknown_accept = float((unknown_preds != 2).mean())
        FAR_unknown_accept_in = float((pred_D != 2).mean())
        FAR_unknown_accept_rx = float((pred_E != 2).mean())
        FAR_unknown_accept_day = float((pred_F != 2).mean())

        miss_drift_rx = float((pred_B != 1).mean())
        miss_drift_day = float((pred_C != 1).mean())
        miss_drift_all = float((drift_preds != 1).mean())

        unknown_score_A = probs_A[:, 2]
        unknown_score_D = probs_D[:, 2]
        unknown_score_allU = np.concatenate([probs_D[:, 2], probs_E[:, 2], probs_F[:, 2]])
        drift_score_B = probs_B[:, 1]
        drift_score_C = probs_C[:, 1]
        drift_score_all = np.concatenate([probs_B[:, 1], probs_C[:, 1]])
        drift_score_A = probs_A[:, 1]

        auc_unknown_in = roc_auc(
            np.concatenate([np.zeros_like(unknown_score_A, dtype=np.int64), np.ones_like(unknown_score_D, dtype=np.int64)]),
            np.concatenate([unknown_score_A, unknown_score_D])
        )
        auc_unknown_all = roc_auc(
            np.concatenate([np.zeros_like(unknown_score_A, dtype=np.int64), np.ones_like(unknown_score_allU, dtype=np.int64)]),
            np.concatenate([unknown_score_A, unknown_score_allU])
        )
        auc_drift_rx = roc_auc(
            np.concatenate([np.zeros_like(drift_score_A, dtype=np.int64), np.ones_like(drift_score_B, dtype=np.int64)]),
            np.concatenate([drift_score_A, drift_score_B])
        )
        auc_drift_day = roc_auc(
            np.concatenate([np.zeros_like(drift_score_A, dtype=np.int64), np.ones_like(drift_score_C, dtype=np.int64)]),
            np.concatenate([drift_score_A, drift_score_C])
        )
        auc_drift_all = roc_auc(
            np.concatenate([np.zeros_like(drift_score_A, dtype=np.int64), np.ones_like(drift_score_all, dtype=np.int64)]),
            np.concatenate([drift_score_A, drift_score_all])
        )

        routing_stats = {
            "p_known_stable_mean": float(aux_A["p_known"].mean()),
            "p_known_drift_mean": float(np.concatenate([aux_B["p_known"], aux_C["p_known"]]).mean()),
            "p_known_unknown_mean": float(np.concatenate([aux_D["p_known"], aux_E["p_known"], aux_F["p_known"]]).mean()),
            "p_drift_cond_stable_mean": float(aux_A["p_drift_cond"].mean()),
            "p_drift_cond_drift_mean": float(np.concatenate([aux_B["p_drift_cond"], aux_C["p_drift_cond"]]).mean()),
            "p_drift_cond_unknown_mean": float(np.concatenate([aux_D["p_drift_cond"], aux_E["p_drift_cond"], aux_F["p_drift_cond"]]).mean()),
            "p_route_stable_mean": float(probs_A[:, 0].mean()),
            "p_route_drift_on_drift_mean": float(np.concatenate([probs_B[:, 1], probs_C[:, 1]]).mean()),
            "p_route_unknown_on_unknown_mean": float(np.concatenate([probs_D[:, 2], probs_E[:, 2], probs_F[:, 2]]).mean()),
        }
        with open(os.path.join(fold_dir, "routing_stats.json"), "w", encoding="utf-8") as f:
            json.dump(routing_stats, f, indent=2)

        np.savez_compressed(
            os.path.join(fold_dir, "soft_router_scores.npz"),
            probs_A=probs_A, probs_B=probs_B, probs_C=probs_C, probs_D=probs_D, probs_E=probs_E, probs_F=probs_F,
            Sid_A=Sid_A, Sid_B=Sid_B, Sid_C=Sid_C, Sid_D=Sid_D, Sid_E=Sid_E, Sid_F=Sid_F,
            Sdom_A=Sdom_A, Sdom_B=Sdom_B, Sdom_C=Sdom_C, Sdom_D=Sdom_D, Sdom_E=Sdom_E, Sdom_F=Sdom_F,
        )

        metrics = dict(
            protocol_tag=protocol_tag,
            rx_protocol=rx_protocol_tag,
            tx_protocol=tx_protocol_tag,
            split=split_id,
            fold=fold,
            KNOWN_TX=KNOWN_TX,
            UNKNOWN_TX=UNKNOWN_TX,
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            acc=acc,
            stable_acc_gate=stable_acc_gate,
            drift_acc_rx=drift_acc_rx,
            drift_acc_day=drift_acc_day,
            drift_acc_all=drift_acc_all,
            unknown_reject_in=unknown_reject_in,
            unknown_reject_rx=unknown_reject_rx,
            unknown_reject_day=unknown_reject_day,
            unknown_reject_all=unknown_reject_all,
            auc_unknown_in=auc_unknown_in,
            auc_unknown_all=auc_unknown_all,
            auc_drift_rx=auc_drift_rx,
            auc_drift_day=auc_drift_day,
            auc_drift_all=auc_drift_all,
            tau_id=tau_id,
            tau_dom=tau_dom,
            temp_id=temp_id,
            temp_dom=temp_dom,
            FRR_stable=FRR_stable,
            false_drift_stable=false_drift_stable,
            false_unknown_stable=false_unknown_stable,
            FAR_unknown_accept=FAR_unknown_accept,
            FAR_unknown_accept_in=FAR_unknown_accept_in,
            FAR_unknown_accept_rx=FAR_unknown_accept_rx,
            FAR_unknown_accept_day=FAR_unknown_accept_day,
            miss_drift_rx=miss_drift_rx,
            miss_drift_day=miss_drift_day,
            miss_drift_all=miss_drift_all,
            p_known_stable_mean=routing_stats["p_known_stable_mean"],
            p_known_drift_mean=routing_stats["p_known_drift_mean"],
            p_known_unknown_mean=routing_stats["p_known_unknown_mean"],
            p_drift_cond_stable_mean=routing_stats["p_drift_cond_stable_mean"],
            p_drift_cond_drift_mean=routing_stats["p_drift_cond_drift_mean"],
            p_drift_cond_unknown_mean=routing_stats["p_drift_cond_unknown_mean"],
            p_route_stable_mean=routing_stats["p_route_stable_mean"],
            p_route_drift_on_drift_mean=routing_stats["p_route_drift_on_drift_mean"],
            p_route_unknown_on_unknown_mean=routing_stats["p_route_unknown_on_unknown_mean"],
            confusion_matrix=cm.tolist(),
        )
        with open(os.path.join(fold_dir, "metrics_gate.json"), "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)

        print(
            f"[{protocol_tag} split {split_id} fold {fold}] "
            f"acc={acc:.3f} stable={stable_acc_gate:.3f} driftRX={drift_acc_rx:.3f} driftDAY={drift_acc_day:.3f} "
            f"unkRej={unknown_reject_all:.3f} FARunk={FAR_unknown_accept:.3f} missDr={miss_drift_all:.3f} "
            f"AUCu={auc_unknown_all:.3f} AUCd={auc_drift_all:.3f}"
        )
        rows.append(metrics)

    return rows

all_rows = []
protocol_specs = []

for spec in PROTOCOL_SPECS:
    protocol_tag = spec["protocol_tag"]
    rx_protocol_tag = spec["rx_protocol"]
    tx_protocol_tag = spec["tx_protocol"]
    source_rxs = list(spec["source_rxs"])
    drift_rx = list(spec["drift_rx"])
    tx_splits = list(spec["tx_splits"])

    protocol_specs.append(spec)

    print("\n===", protocol_tag, "===")
    for split in tx_splits:
        split_id = int(split["split_id"]) if isinstance(split, dict) and "split_id" in split else (tx_splits.index(split) + 1)
        if isinstance(split, dict):
            KNOWN_TX = split["known"]
            UNKNOWN_TX = split["unknown"]
        else:
            KNOWN_TX, UNKNOWN_TX = split
        print(f"[split {split_id}] KNOWN_TX={KNOWN_TX} UNKNOWN_TX={UNKNOWN_TX}")
        all_rows.extend(run_one_split(
            protocol_tag=protocol_tag,
            rx_protocol_tag=rx_protocol_tag,
            tx_protocol_tag=tx_protocol_tag,
            split_id=split_id,
            KNOWN_TX=KNOWN_TX,
            UNKNOWN_TX=UNKNOWN_TX,
            source_rxs=source_rxs,
            drift_rx=drift_rx,
        ))

with open(os.path.join(RUN_DIR, "protocol_specs.json"), "w", encoding="utf-8") as f:
    json.dump(protocol_specs, f, indent=2)
with open(os.path.join(RUN_DIR, "metrics_all_rows.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)




=== RX3-9_TX2-4 ===
[split 1] KNOWN_TX=['20-15', '8-20'] UNKNOWN_TX=['14-10', '14-7', '20-19', '6-15']
[RX3-9_TX2-4 split 1 fold 1] acc=0.998 stable=0.904 driftRX=0.170 driftDAY=0.047 unkRej=0.544 FARunk=0.456 missDr=0.861 AUCu=0.905 AUCd=0.530
[RX3-9_TX2-4 split 1 fold 2] acc=1.000 stable=0.899 driftRX=0.254 driftDAY=0.082 unkRej=0.357 FARunk=0.643 missDr=0.789 AUCu=0.802 AUCd=0.541
[RX3-9_TX2-4 split 1 fold 3] acc=0.996 stable=0.897 driftRX=0.119 driftDAY=0.045 unkRej=0.909 FARunk=0.091 missDr=0.899 AUCu=0.967 AUCd=0.419
[RX3-9_TX2-4 split 1 fold 4] acc=0.978 stable=0.902 driftRX=0.250 driftDAY=0.106 unkRej=0.659 FARunk=0.341 missDr=0.786 AUCu=0.929 AUCd=0.531
[RX3-9_TX2-4 split 1 fold 5] acc=0.946 stable=0.928 driftRX=0.109 driftDAY=0.039 unkRej=0.217 FARunk=0.783 missDr=0.908 AUCu=0.845 AUCd=0.611
[split 2] KNOWN_TX=['14-7', '6-15'] UNKNOWN_TX=['14-10', '20-15', '20-19', '8-20']
[RX3-9_TX2-4 split 2 fold 1] acc=0.996 stable=0.895 driftRX=0.133 driftDAY=0.011 unkRej=0.635 FARunk=0.

In [10]:
# =========================
# Summary
# =========================
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

METRIC_KEYS = [
    "acc",
    "stable_acc_gate",
    "drift_acc_rx",
    "drift_acc_day",
    "drift_acc_all",
    "unknown_reject_in",
    "unknown_reject_rx",
    "unknown_reject_day",
    "unknown_reject_all",
    "auc_unknown_in",
    "auc_unknown_all",
    "auc_drift_rx",
    "auc_drift_day",
    "auc_drift_all",
    "FRR_stable",
    "false_drift_stable",
    "false_unknown_stable",
    "FAR_unknown_accept",
    "FAR_unknown_accept_in",
    "FAR_unknown_accept_rx",
    "FAR_unknown_accept_day",
    "miss_drift_rx",
    "miss_drift_day",
    "miss_drift_all",
    "tau_id",
    "tau_dom",
    "temp_id",
    "temp_dom",
    "p_known_stable_mean",
    "p_known_drift_mean",
    "p_known_unknown_mean",
    "p_drift_cond_stable_mean",
    "p_drift_cond_drift_mean",
    "p_drift_cond_unknown_mean",
    "p_route_stable_mean",
    "p_route_drift_on_drift_mean",
    "p_route_unknown_on_unknown_mean",
]

def summarize_rows(rows, metric_keys=METRIC_KEYS):
    summary = {"n_rows": int(len(rows))}
    for key in metric_keys:
        m, s = mean_std([r[key] for r in rows])
        summary[key] = {"mean": m, "std": s}

    cm_sum = np.zeros((3, 3), dtype=np.int64)
    for r in rows:
        cm_sum += np.array(r["confusion_matrix"], dtype=np.int64)
    summary["confusion_matrix_sum"] = cm_sum.tolist()
    return summary

def summarize_by_group(rows, group_key, metric_keys=METRIC_KEYS):
    groups = {}
    for r in rows:
        groups.setdefault(r[group_key], []).append(r)
    out = {}
    for key, group_rows in groups.items():
        out[key] = summarize_rows(group_rows, metric_keys=metric_keys)
    return out

summary = summarize_rows(all_rows, metric_keys=METRIC_KEYS)
summary_by_protocol = summarize_by_group(all_rows, "protocol_tag", metric_keys=METRIC_KEYS)
summary_by_rx_protocol = summarize_by_group(all_rows, "rx_protocol", metric_keys=METRIC_KEYS)
summary_by_tx_protocol = summarize_by_group(all_rows, "tx_protocol", metric_keys=METRIC_KEYS)

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
with open(os.path.join(RUN_DIR, "summary_by_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(summary_by_protocol, f, indent=2)
with open(os.path.join(RUN_DIR, "summary_by_rx_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(summary_by_rx_protocol, f, indent=2)
with open(os.path.join(RUN_DIR, "summary_by_tx_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(summary_by_tx_protocol, f, indent=2)

print("=== Overall mean ± std over protocol × split × fold ===")
for k, v in summary.items():
    if k in ["n_rows", "confusion_matrix_sum"]:
        continue
    print(f"{k:28s}: {v['mean']:.3f} ± {v['std']:.3f}")

print("\nConfusion matrix SUM (rows=GT, cols=Pred) [Stable, Drift, Unknown]:")
print(np.array(summary["confusion_matrix_sum"], dtype=np.int64))

print("\n=== Per-protocol quick view ===")
for protocol_tag in sorted(summary_by_protocol.keys()):
    s = summary_by_protocol[protocol_tag]
    print(
        f"[{protocol_tag}] "
        f"FARunk={s['FAR_unknown_accept']['mean']:.3f} ± {s['FAR_unknown_accept']['std']:.3f} | "
        f"missRX={s['miss_drift_rx']['mean']:.3f} ± {s['miss_drift_rx']['std']:.3f} | "
        f"missDAY={s['miss_drift_day']['mean']:.3f} ± {s['miss_drift_day']['std']:.3f} | "
        f"AUCuAll={s['auc_unknown_all']['mean']:.3f} ± {s['auc_unknown_all']['std']:.3f} | "
        f"AUCdAll={s['auc_drift_all']['mean']:.3f} ± {s['auc_drift_all']['std']:.3f}"
    )

print("\nSaved overall summary to:", os.path.join(RUN_DIR, "summary_mean_std.json"))
print("Saved per-protocol summary to:", os.path.join(RUN_DIR, "summary_by_protocol.json"))
print("Saved RX-family summary to:", os.path.join(RUN_DIR, "summary_by_rx_protocol.json"))
print("Saved TX-family summary to:", os.path.join(RUN_DIR, "summary_by_tx_protocol.json"))



=== Overall mean ± std over protocol × split × fold ===
acc                         : 0.996 ± 0.005
stable_acc_gate             : 0.894 ± 0.010
drift_acc_rx                : 0.164 ± 0.132
drift_acc_day               : 0.100 ± 0.067
drift_acc_all               : 0.135 ± 0.085
unknown_reject_in           : 0.582 ± 0.242
unknown_reject_rx           : 0.613 ± 0.236
unknown_reject_day          : 0.579 ± 0.227
unknown_reject_all          : 0.598 ± 0.225
auc_unknown_in              : 0.836 ± 0.109
auc_unknown_all             : 0.843 ± 0.105
auc_drift_rx                : 0.511 ± 0.137
auc_drift_day               : 0.545 ± 0.090
auc_drift_all               : 0.521 ± 0.107
FRR_stable                  : 0.106 ± 0.010
false_drift_stable          : 0.043 ± 0.005
false_unknown_stable        : 0.063 ± 0.012
FAR_unknown_accept          : 0.402 ± 0.225
FAR_unknown_accept_in       : 0.418 ± 0.242
FAR_unknown_accept_rx       : 0.387 ± 0.236
FAR_unknown_accept_day      : 0.421 ± 0.227
miss_drift_rx       